In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pickle
import json

In [43]:
adults = pd.read_csv("../data/adult.csv")
adults.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9.0,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9.0,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10.0,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4.0,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,2646x63,Some-college,10.0,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [44]:
num_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

for col in num_cols:
    adults[col] = pd.to_numeric(adults[col].astype(str).str.replace(r'[^0-9]', '', regex=True), errors='coerce')

print(adults.dtypes)

age               float64
workclass          object
fnlwgt            float64
education          object
education.num     float64
marital.status     object
occupation         object
relationship       object
race               object
sex                object
capital.gain      float64
capital.loss        int64
hours.per.week    float64
native.country     object
income             object
dtype: object


In [45]:
current_max = adults['education.num'].max()

if pd.notnull(current_max) and current_max != 16 and current_max > 0:
    scaling_factor = current_max / 16
    adults['education.num'] = adults['education.num'] / scaling_factor

adults['education.num'] = adults['education.num'].round().astype('Int64')

edu_map = adults.dropna(subset=['education']).set_index('education.num')['education'].to_dict()
adults['education'] = adults['education'].fillna(adults['education.num'].map(edu_map))

adults['age'] = adults['age'].clip(upper=85).astype('Int64')

adults = adults[adults['hours.per.week'] <= 99].copy()
adults['hours.per.week'] = adults['hours.per.week'].astype('Int64')

adults['is_capital_maxed'] = (adults['capital.gain'] >= 99990).astype('Int64')
adults['capital_gain_log'] = np.log1p(adults['capital.gain'])

adults = adults.dropna(subset=['education.num'])

print(f"Max Age: {adults['age'].max()}")
print(f"Max Hours: {adults['hours.per.week'].max()}")
print(f"Education Range: {adults['education.num'].min()} to {adults['education.num'].max()}")
print(f"Missing Education Numbers: {adults['education.num'].isnull().sum()}")
print(f"Final Row Count: {len(adults)}")

Max Age: 85
Max Hours: 98
Education Range: 1 to 16
Missing Education Numbers: 0
Final Row Count: 32474


In [46]:
adults_clean = adults.dropna()
adults_clean.isnull().sum()

age                 0
workclass           0
fnlwgt              0
education           0
education.num       0
marital.status      0
occupation          0
relationship        0
race                0
sex                 0
capital.gain        0
capital.loss        0
hours.per.week      0
native.country      0
income              0
is_capital_maxed    0
capital_gain_log    0
dtype: int64

In [47]:
adults_clean = adults_clean.replace('?', 'Unknown')

cols_to_check = ['workclass', 'occupation', 'native.country']
for col in cols_to_check:
    unknown_count = (adults_clean[col] == 'Unknown').sum()
    print(f"{col}: {unknown_count} 'Unknown' values")

workclass: 1825 'Unknown' values
occupation: 1832 'Unknown' values
native.country: 580 'Unknown' values


In [48]:
# Define the feature set
ethical_features = [
    'age', 'workclass', 'education.num', 'occupation',
    'capital.gain', 'capital.loss', 'hours.per.week',
    'capital_gain_log', 'is_capital_maxed'
]

# Select the data
X_ethical = adults_clean[ethical_features].copy()
y_ethical = adults_clean['income'].apply(lambda x: 1 if '>50K' in str(x) else 0)

# One-hot encode categorical variables
X_ethical_final = pd.get_dummies(X_ethical, columns=['workclass', 'occupation'], drop_first=True, dtype=int)


In [49]:
# Splitsen in train en test
X_train_eth, X_test_eth, y_train_eth, y_test_eth = train_test_split(
    X_ethical_final, y_ethical, test_size=0.2, random_state=42
)

In [50]:
X_test_eth.to_csv('../data/X_test.csv')

In [51]:
# Scale the features
scaler_eth = StandardScaler()
X_train_eth_scaled = scaler_eth.fit_transform(X_train_eth)
X_test_eth_scaled = scaler_eth.transform(X_test_eth)


In [52]:
# Define the Random Forest with the previously found optimal hyperparameters
ethical_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    max_features='sqrt',
    random_state=42
)

# Train the ethical model
ethical_rf.fit(X_train_eth_scaled, y_train_eth)


,n_estimators,200
,criterion,'gini'
,max_depth,20
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [53]:
with open("../prediction_pkg/model.pkl", "wb") as f:
    pickle.dump(ethical_rf, f)

In [54]:
with open("../prediction_pkg/scaler.pkl", "wb") as f:
    pickle.dump(scaler_eth, f)